# Quick Dask memory usage and repartitioning

This runnable example answers the question: **How do I do a quick look at memory usage for optimising?**

The notebook walks through a small astrometry-themed workflow:

- Attach to a local Dask client.
- Build a synthetic star catalog with pandas.
- Measure baseline dataframe memory usage.
- Apply a simple memory optimization with downcasting.
- Measure Dask dataframe memory by partition.
- Use those measurements to choose a rough repartitioning strategy.

The dataset is intentionally synthetic so the example is quick to run and easy to adapt. For real catalogs, repeat the measurements after loading representative data and before committing to a partitioning strategy.


## Imports and setup

This notebook uses pandas to create a small synthetic catalog, Dask to partition it, and `distributed.Client` to run the Dask computations locally. The `math` import is used later for a simple partition-count heuristic.


In [ ]:
import numpy as np
import pandas as pd
import math
import dask.dataframe as dd
from distributed import Client


## Start a local Dask client

A local client is enough for this minimal example and exposes the same Dask dataframe APIs used on a larger cluster. Replace `Client()` with `Client("address:port")` if you already have a scheduler running.


In [ ]:
# 1) Attach to a Dask client (local cluster)
client = Client()  # or Client('address:port') for an existing cluster
print("Dask client:", client)


## Create a synthetic star catalog

The example builds a pandas dataframe with common astrometry-like columns. Starting in pandas makes it easy to inspect baseline memory usage before converting to a Dask dataframe.


In [ ]:

# 2) Build synthetic astrometry of stars dataset
N = 200_000  # adjust as needed
rng = np.random.default_rng(seed=123)

stars_pd = pd.DataFrame({
    'source_id': np.arange(N, dtype=np.int32),
    'ra': rng.random(N) * 360.0,                      # 0-360 degrees
    'dec': rng.random(N) * 180.0 - 90.0,              # -90 to +90
    'parallax': rng.exponential(scale=10.0, size=N),  # milliarcseconds
    'pmra_cosdec': rng.normal(loc=0.0, scale=5.0, size=N),
    'pmdec': rng.normal(loc=0.0, scale=5.0, size=N),
    'phot_g_mean_mag': rng.normal(loc=15.0, scale=3.0, size=N),
    'bp_rp': rng.normal(loc=1.0, scale=0.5, size=N),})

## Measure the pandas baseline

`memory_usage(deep=True)` reports the in-memory footprint of the dataframe. This gives a baseline for comparing later dtype changes and partition-level measurements.


In [ ]:

# 3) Baseline memory usage (deep=True to include Python objects)
size_bytes = stars_pd.memory_usage(deep=True).sum()
size_mb = size_bytes / 1024**2
print(f"Baseline stars_pd size: {size_mb:.2f} MB")

## Downcast columns where precision allows

Downcasting floating-point columns can reduce memory usage substantially. In a real workflow, check whether reduced precision is scientifically acceptable before applying this broadly.


In [ ]:



# 4) Simple memory optimization

# - downcast numeric columns where possible
stars_pd['phot_g_mean_mag'] = pd.to_numeric(stars_pd['phot_g_mean_mag'], downcast='float')
stars_pd['parallax'] = pd.to_numeric(stars_pd['parallax'], downcast='float')
stars_pd['ra'] = stars_pd['ra'].astype('float32')
stars_pd['dec'] = stars_pd['dec'].astype('float32')

size_bytes_opt = stars_pd.memory_usage(deep=True).sum()
size_mb_opt = size_bytes_opt / 1024**2
print(f"After optimization: {size_mb_opt:.2f} MB")
print(f"Memory reduction: {size_mb - size_mb_opt:.2f} MB")



## Measure memory by Dask partition

After converting to a Dask dataframe, map the same memory calculation across partitions. This helps identify whether partitions are likely to be too large, too small, or unevenly sized.


In [ ]:
# 5) Simple Dask variant: per-partition memory usage
ddf = dd.from_pandas(stars_pd, npartitions=4)
per_part_mem = ddf.map_partitions(lambda df: df.memory_usage(deep=True).sum(), meta=('mem', 'int64')).compute()
print("Per-partition memory usage (bytes):", per_part_mem.tolist())
print("Sum of per-partition mem (approx total):", int(per_part_mem.sum()))



## Close the first local client

The first section is self-contained, so the local Dask client can be closed after the quick memory check. If you are adapting the notebook interactively, you can leave the client open until the end of your session.


In [ ]:
# Cleanup (optional)
client.close()

## From measurement to action

The previous cells show how to measure total dataframe memory and approximate memory per Dask partition. The next step is to turn those measurements into a practical recommendation for repartitioning.

Typical questions to answer:

- How big is a single partition? If a partition cannot fit comfortably in worker memory, use more partitions, smaller chunks, or a different partitioning key.
- How many partitions should the dataframe have? A common starting point is to aim for a per-task partition size that fits in memory with headroom, such as 100-300 MB per partition, depending on available RAM and workload shape.
- Would a meaningful partitioning key help? Partitioning by a column used in filters, such as RA bins, can improve query locality, but may require an expensive shuffle up front.

A simple workflow is:

1. Measure per-partition memory usage for the current dataframe.
2. Choose a target memory size per partition.
3. Estimate a new partition count from the measured size.
4. Repartition and re-measure.
5. Optionally repartition by an analysis key when later queries will benefit.


Notes and cautions
- Measuring per-partition memory via memory_usage(deep=True) is a practical proxy, but actual runtime memory depends on tasks in flight and scheduler behavior. Use a LocalCluster with a reasonable memory_limit per worker if you can.
- Repartitioning by index (set_index) involves a shuffle and is relatively expensive; use it when you’ll actually benefit from sorted/partitioned by that key (e.g., frequent range filters on ra_bin).
- A good starting point for large datasets is to target around a few hundred MB per partition, tuned to your cluster’s memory and the number of workers.
- If you have a real cluster, consider setting a memory_limit per worker and using a LocalCluster or a distributed cluster; you can also examine the Dask dashboards for live memory use.
- After repartitioning, re-run the per-partition memory check to validate that the new partition sizes align with your target.

## Estimate a partition count from measured memory

This section turns the per-partition memory measurements into a rough repartitioning recommendation. The target size is a heuristic, not a universal rule: choose a value that leaves comfortable headroom for your workers and the operations you plan to run.


In [ ]:
# Create Dask dataframe from the optimized pandas df
ddf = dd.from_pandas(stars_pd, npartitions=4)

# 1) measure memory per partition (approximate)
per_part_mem = ddf.map_partitions(
    lambda df: df.memory_usage(deep=True).sum(),
    meta=('mem', 'int64')
).compute()

print("Per-partition memory (bytes):", per_part_mem.tolist())
avg_per_part = float(np.mean(per_part_mem))
print(f"Avg per-partition memory: {avg_per_part/1024**2:.2f} MB")

# 2) choose a target per-partition size (adjust to your environment)
target_per_part_mb = 150  # e.g., 150 MB per partition
target_per_part_bytes = target_per_part_mb * 1024**2

# 3) compute new number of partitions from heuristic
current_npart = ddf.npartitions
new_npart = max(1, int(math.ceil(current_npart * avg_per_part / target_per_part_bytes)))
print(f"Repartition to {new_npart} partitions for ~{target_per_part_mb} MB/partition")

# 4) apply repartition
ddf_repart = ddf.repartition(npartitions=new_npart)

# Optional: verify memory after repartition on a sample (may not reflect exact in-use memory)
per_part_mem_after = ddf_repart.map_partitions(
    lambda df: df.memory_usage(deep=True).sum(),
    meta=('mem', 'int64')
).compute()
print("Per-partition memory after repartition (bytes):", per_part_mem_after.tolist())



## Optional: partition by an analysis key

When downstream work repeatedly filters or groups by sky position, adding an RA bin can make access patterns more local. Setting the bin as the index causes a shuffle, so this is worth doing only when later queries benefit from that organization.


In [ ]:
# If you plan to filter by RA or do bin-based analysis, repartition by a key can help.
# Example: create RA-bin (e.g., 10-degree bins) and set as index (causes shuffle)
def add_ra_bin(df):
    import numpy as np
    df = df.copy()
    bin_size = 10.0  # 10-degree bins
    df['ra_bin'] = (np.floor(df['ra'] / bin_size) * bin_size).astype(np.float64)
    return df

# Add the bucket column; we need to set proper meta
ddf_with_bin = ddf.map_partitions(add_ra_bin, meta=ddf._meta.assign(ra_bin='float64'))
# Now partition by the new key
# Option 1: set as index for range-partitioned sorting/shuffle-friendly operations
ddf_by_bin = ddf_with_bin.set_index('ra_bin', sorted=True)

# If you want to keep a column instead of index, you can:
# ddf_by_bin = ddf_with_bin.repartition(partition_size="100MB")  # an alternative approach